# Gemma 4 Eval — in-process with lighteval

Drop-in notebook for any Gemma 4 project. Plug two model paths in the settings
cell below, run the notebook, get a comparable score plus a detail dashboard.

**Default run**: Google's official Gemma 4 E2B-it vs E4B-it as a smoke test.
Change `TEST_MODEL_PATH` to your own fine-tune (or another Gemma 4 variant)
to compare behaviour.

Design notes:
- No subprocess, no writefile. Lighteval's `Pipeline` runs in-process, so
  errors land as Python tracebacks and every intermediate tensor is yours
  to inspect.
- On Kaggle T4 x2 the base and test models load on separate GPUs and run
  in parallel threads.
- Raise `ROUNDS` to 8 for an 8-PAC-style multi-sample comparison.
- Everything about the custom model wrapper lives in a single cell below,
  readable and editable.


In [ ]:
# User settings — the only cell most runs need to touch.

# Model paths. On Kaggle, attach the official Google Gemma 4 model inputs or
# a custom model and point these at the resulting /kaggle/input directories.
BASE_MODEL_PATH = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1'
TEST_MODEL_PATH = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1'

# Optional KaggleHub fallback — if the paths above don't exist locally, pull
# the official models from KaggleHub so the smoke test still runs.
USE_KAGGLEHUB_FALLBACK = True
BASE_KAGGLEHUB_MODEL = 'google/gemma-4/transformers/gemma-4-e2b-it'
TEST_KAGGLEHUB_MODEL = 'google/gemma-4/transformers/gemma-4-e4b-it'

# Evaluation shape.
TASK = 'lighteval|mmlu_pro|0|0'   # lighteval task spec: suite|name|few-shot|truncate
N_QUESTIONS = 1                   # tiny smoke; bump for real runs
ROUNDS = 1                        # 8 for 8-PAC-style sample variance
SAMPLES_START = 0

# Generation — Gemma 4 calibrated sampling.
MAX_NEW_TOKENS = 4096
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 64

# Device/dtype. 'auto' picks bf16 on Ampere+, fp16 on T4/V100.
DEVICE_MAP = 'auto'
DTYPE = 'auto'

# Parallel dual-GPU path. On Kaggle T4 x2 we pin base->cuda:0 and test->cuda:1
# and run them in threads. Set to False to serialise (one card at a time).
PARALLEL_TWO_GPU = True

# Run name + output.
RUN_NAME = 'gemma4-e2b-vs-e4b'
OUTPUT_ROOT = None   # None -> /kaggle/working/<RUN_NAME> on Kaggle, ./runs/<RUN_NAME> locally

# Notebook knobs.
RUN_INSTALL = None    # None -> auto-detect (install on Kaggle, skip locally if already set up)
RUN_EVAL = True       # False to stop after setup and inspect resolved config
VISUAL_MAX_QUESTIONS = 8
VISUAL_TEXT_PREVIEW_CHARS = 420


## 1. Install & import

In [ ]:
# Install dependencies. Auto-runs on Kaggle; skip flag available above.
import sys, subprocess
from pathlib import Path

IN_KAGGLE = Path('/kaggle/working').exists()
if RUN_INSTALL is None:
    RUN_INSTALL = IN_KAGGLE

packages = [
    'lighteval @ git+https://github.com/LetheanNetwork/lighteval.git@gemma4',
    'transformers', 'accelerate', 'safetensors',
    'pandas', 'pyarrow', 'tqdm', 'plotly', 'kagglehub',
]
try:
    import torch  # noqa: F401
except ImportError:
    packages.append('torch')

if RUN_INSTALL:
    print('Installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])
    print('Done.')
else:
    print('Skipped (set RUN_INSTALL=True to force).')


In [ ]:
# Imports + device diagnostics.
import os, sys, json, logging, platform, shutil, warnings, datetime as dt
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
from IPython.display import HTML, display

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('accelerate').setLevel(logging.ERROR)
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

print(f'python {sys.version.split()[0]}  |  torch {torch.__version__}  |  cuda={torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    cc = torch.cuda.get_device_capability(i)
    print(f'  cuda:{i}  {p.name}  cc={cc[0]}.{cc[1]}  {p.total_memory/1024**3:.1f} GB')

parallel_two_gpu = bool(PARALLEL_TWO_GPU and num_gpus >= 2)
if parallel_two_gpu:
    print('Dual-GPU path enabled.')
elif num_gpus:
    print('Single-GPU path (sequential base then test).')
else:
    print('CPU only — inference will be very slow; change to a GPU runtime.')

# Resolve output root.
if OUTPUT_ROOT:
    run_dir = Path(OUTPUT_ROOT) / RUN_NAME
elif IN_KAGGLE:
    run_dir = Path('/kaggle/working') / RUN_NAME
else:
    run_dir = Path('runs') / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
print(f'results -> {run_dir}')


## 2. Resolve models

In [ ]:
# Resolve model paths, with optional KaggleHub fallback for the defaults.
def resolve_model_path(local_path, kagglehub_model, label):
    p = Path(local_path)
    if p.exists():
        print(f'[{label}] local: {p}')
        return str(p)
    if USE_KAGGLEHUB_FALLBACK:
        import kagglehub
        print(f'[{label}] local not found, pulling {kagglehub_model} from KaggleHub...')
        resolved = kagglehub.model_download(kagglehub_model)
        print(f'[{label}] resolved: {resolved}')
        return resolved
    raise FileNotFoundError(f'[{label}] {local_path} not present and KaggleHub fallback disabled.')

base_model = resolve_model_path(BASE_MODEL_PATH, BASE_KAGGLEHUB_MODEL, 'base')
test_model = resolve_model_path(TEST_MODEL_PATH, TEST_KAGGLEHUB_MODEL, 'test')


## 3. Lighteval model wrapper (inline — no subprocess, no writefile)

In [ ]:
# Inline lighteval model wrapper. Read it, modify it if you want.
#
# Lighteval's Pipeline accepts any subclass of LightevalModel. We wrap
# transformers AutoModelForCausalLM + AutoProcessor so it works with any
# Gemma 4 checkpoint (official, fine-tune, merged LoRA, etc.).
#
# Gemma 4 is multimodal — the Kaggle-official loader is AutoProcessor, which
# wraps the text tokenizer + image/audio preprocessors. We only use the text
# path here; the processor's tokenizer satisfies lighteval's interface.

from transformers import AutoModelForCausalLM, AutoProcessor
from lighteval.models.abstract_model import LightevalModel
from lighteval.models.model_output import ModelResponse

# Gemma 4 supports a "thinking" mode via chat template. For MCQ-style evals
# (MMLU-Pro etc.) we disable thinking so the response is a direct answer.
ENABLE_THINKING = False


def pick_dtype():
    if DTYPE != 'auto':
        return getattr(torch, DTYPE)
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16


class Gemma4Model(LightevalModel):
    """LightevalModel wrapping transformers Gemma 4 via AutoProcessor."""

    def __init__(self, model_path: str, device_map='auto'):
        self.model_path = model_path
        self.processor = AutoProcessor.from_pretrained(model_path)
        tok = self.processor.tokenizer
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = 'left'
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, dtype=pick_dtype(), device_map=device_map,
        )
        self.model.train(False)  # inference mode (== model.eval())
        self.device = next(self.model.parameters()).device

    # --- Abstract properties required by LightevalModel ---
    @property
    def tokenizer(self):
        return self.processor.tokenizer

    @property
    def max_length(self):
        return getattr(self.model.config, 'max_position_embeddings', 8192)

    @property
    def add_special_tokens(self):
        # Chat template already adds the required special tokens, so our
        # manual tok_encode path should not add BOS again.
        return False

    # --- The only task entrypoint MMLU-Pro needs ---
    def greedy_until(self, docs, **kwargs):
        responses = []
        for doc in docs:
            messages = [{'role': 'user', 'content': doc.query}]
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=ENABLE_THINKING,
            )
            inputs = self.processor(text=text, return_tensors='pt').to(self.device)
            input_len = inputs['input_ids'].shape[-1]
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                    pad_token_id=self.processor.tokenizer.eos_token_id,
                )
            gen = self.processor.decode(out[0][input_len:], skip_special_tokens=True)
            responses.append(ModelResponse(text=[gen]))
        return responses

    # Lighteval only calls these for specific task types. MMLU-Pro uses
    # greedy_until via extractive_match, so leaving these as clear errors.
    def loglikelihood(self, *args, **kwargs):
        raise NotImplementedError('Gemma4Model does not implement loglikelihood (pick a greedy_until-based task).')
    def loglikelihood_rolling(self, *args, **kwargs):
        raise NotImplementedError('Gemma4Model does not implement loglikelihood_rolling.')


print('Gemma4Model wrapper defined.')


## 4. Run evaluation

In [ ]:
# Run the evaluation with lighteval's in-process Pipeline.
from lighteval.pipeline import Pipeline, PipelineParameters, ParallelismManager
from lighteval.logging.evaluation_tracker import EvaluationTracker


def run_one(model_path: str, side: str, round_idx: int, gpu_index):
    """Evaluate a single model for one round. Returns the details parquet path."""
    dm = f'cuda:{gpu_index}' if gpu_index is not None else DEVICE_MAP
    out_dir = run_dir / f'{side}_round{round_idx}'
    if out_dir.exists():
        shutil.rmtree(out_dir)

    tracker = EvaluationTracker(output_dir=str(out_dir), save_details=True)
    params = PipelineParameters(
        launcher_type=ParallelismManager.NONE,
        max_samples=N_QUESTIONS,
        samples_start=SAMPLES_START,
    )
    model = Gemma4Model(model_path, device_map=dm)

    print(f'[{side}] round {round_idx}/{ROUNDS}  path={model_path}  device={dm}')
    pipeline = Pipeline(
        tasks=TASK,
        pipeline_parameters=params,
        evaluation_tracker=tracker,
        model=model,
    )
    pipeline.evaluate()
    pipeline.save_and_push_results()

    # Free GPU memory before next round/side.
    del model, pipeline
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    parquets = sorted(out_dir.glob('details/**/*.parquet'))
    if not parquets:
        raise RuntimeError(f'[{side}] no details parquet produced in {out_dir}')
    return str(parquets[0])


def run_paired_rounds():
    """Run base + test across ROUNDS. Uses dual-GPU parallel path when available."""
    base_paths, test_paths = [], []
    for round_idx in range(1, ROUNDS + 1):
        if parallel_two_gpu:
            with ThreadPoolExecutor(max_workers=2) as ex:
                fb = ex.submit(run_one, base_model, 'base', round_idx, 0)
                ft = ex.submit(run_one, test_model, 'test', round_idx, 1)
                base_paths.append(fb.result())
                test_paths.append(ft.result())
        else:
            single_gpu = 0 if num_gpus else None
            base_paths.append(run_one(base_model, 'base', round_idx, single_gpu))
            test_paths.append(run_one(test_model, 'test', round_idx, single_gpu))
    return base_paths, test_paths


if RUN_EVAL:
    base_detail_paths, test_detail_paths = run_paired_rounds()
    print()
    print('Round detail parquets:')
    for side, paths in [('base', base_detail_paths), ('test', test_detail_paths)]:
        for i, p in enumerate(paths, 1):
            print(f'  {side} r{i}: {p}')
else:
    print('RUN_EVAL is False — stop here to inspect resolved config, then set True and run.')


## 5. Parse results + comparison

In [ ]:
# Parse lighteval detail parquet files and build a compact comparison report.

def first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return p
    return None


def extract_text(resp):
    try:
        text = resp['text']
    except Exception:
        text = getattr(resp, 'text', resp)
    if isinstance(text, (list, tuple)):
        return str(text[0]) if text else ''
    try:
        values = list(text)
        return str(values[0]) if values else ''
    except Exception:
        return str(text)


def extract_answer(text):
    m = re.search(r'Answer:\s*([A-Z])', text)
    if m:
        return m.group(1)
    m = re.search(r'\b([A-J])\b', text)
    return m.group(1) if m else '?'


def metric_hit(metric):
    if isinstance(metric, dict):
        for key in ('extractive_match', 'exact_match', 'accuracy'):
            if key in metric:
                return int(metric[key])
    return 0


def choice_map_from_query(query):
    return dict(re.findall(r'^([A-Z]):\s*(.+?)$', str(query), re.MULTILINE))


def analyze_pair(base_paths, test_paths):
    ref_path = first_existing(base_paths + test_paths)
    if not ref_path:
        raise RuntimeError('No lighteval detail parquet files were produced. Check the previous cell output.')

    ref_df = pd.read_parquet(ref_path)
    base_dfs = [pd.read_parquet(p) if p and Path(p).exists() else None for p in base_paths]
    test_dfs = [pd.read_parquet(p) if p and Path(p).exists() else None for p in test_paths]

    rows = []
    question_summaries = []
    for q_idx in range(len(ref_df)):
        doc = ref_df.iloc[q_idx].get('doc', {})
        query = doc.get('query', '') if isinstance(doc, dict) else str(doc)
        gold_idx = doc.get('gold_index', None) if isinstance(doc, dict) else None
        gold_letter = chr(ord('A') + int(gold_idx)) if gold_idx is not None else '?'
        choices = choice_map_from_query(query)
        gold_text = choices.get(gold_letter, '')
        qbody_match = re.search(r'question\.\s*\n+(.*?)(?=\n[A-Z]:)', query, re.DOTALL)
        qbody = qbody_match.group(1).strip() if qbody_match else str(query)[:500]

        per_side = {}
        for side, dfs, model_name in [('base', base_dfs, base_model), ('test', test_dfs, test_model)]:
            answers, hits = [], []
            for round_num, df in enumerate(dfs, start=1):
                if df is None or q_idx >= len(df):
                    continue
                row = df.iloc[q_idx]
                text = extract_text(row.get('model_response', {}))
                answer = extract_answer(text)
                hit = metric_hit(row.get('metric', {}))
                answers.append(answer)
                hits.append(hit)
                rows.append({
                    'run_name': RUN_NAME,
                    'task': TASK,
                    'model_side': side,
                    'model_name': model_name,
                    'round': round_num,
                    'samples_start': samples_start,
                    'question_index': samples_start + q_idx,
                    'question_body': qbody,
                    'gold_letter': gold_letter,
                    'gold_text': gold_text,
                    'extracted_answer': answer,
                    'hit': hit,
                    'full_text': text,
                })
            total = len(hits)
            correct = sum(hits)
            majority = Counter(answers).most_common(1)[0][0] if answers else '?'
            per_side[side] = {
                'answers': answers,
                'hits': hits,
                'correct': correct,
                'total': total,
                'accuracy': correct / total if total else 0.0,
                'majority_answer': majority,
                'majority_hit': int(majority == gold_letter) if gold_letter != '?' else 0,
            }
        question_summaries.append({
            'question_index': samples_start + q_idx,
            'question': qbody,
            'gold_letter': gold_letter,
            'gold_text': gold_text,
            'base': per_side['base'],
            'test': per_side['test'],
        })

    detail_df = pd.DataFrame(rows)
    totals = {}
    for side in ('base', 'test'):
        side_df = detail_df[detail_df.model_side == side]
        total = len(side_df)
        correct = int(side_df.hit.sum()) if total else 0
        q_majority = sum(q[side]['majority_hit'] for q in question_summaries)
        totals[side] = {
            'model': base_model if side == 'base' else test_model,
            'samples': total,
            'correct': correct,
            'per_round_accuracy_pct': round(100 * correct / total, 2) if total else 0.0,
            'majority_correct': int(q_majority),
            'questions': len(question_summaries),
            'majority_accuracy_pct': round(100 * q_majority / len(question_summaries), 2) if question_summaries else 0.0,
        }
    totals['delta_pp'] = round(totals['test']['per_round_accuracy_pct'] - totals['base']['per_round_accuracy_pct'], 2)
    totals['majority_delta_pp'] = round(totals['test']['majority_accuracy_pct'] - totals['base']['majority_accuracy_pct'], 2)
    return detail_df, question_summaries, totals

if RUN_EVAL:
    detail_df, question_summaries, totals = analyze_pair(base_parquets, test_parquets)
    print('Result data parsed. The next cell renders the educational dashboard.')
    print(f"base per-round: {totals['base']['per_round_accuracy_pct']:.2f}% | test per-round: {totals['test']['per_round_accuracy_pct']:.2f}% | delta: {totals['delta_pp']:+.2f} pp")
else:
    detail_df = pd.DataFrame()
    question_summaries = []
    totals = {}


## 6. Visual dashboard

In [ ]:
# Educational visual dashboard: show the data slices, votes, and deltas.

PLOT_COLORS = {
    'base': '#2f6fed',
    'test': '#19a974',
    'right': '#19a974',
    'miss': '#e5484d',
    'unknown': '#d5a100',
}


def pretty_model_name(model_path):
    parts = Path(str(model_path)).parts
    if 'transformers' in parts:
        idx = parts.index('transformers')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    if len(parts) >= 2 and parts[-1].isdigit():
        return parts[-2]
    return Path(str(model_path)).name or str(model_path)


def clamp_pct(value):
    try:
        return max(0.0, min(100.0, float(value)))
    except Exception:
        return 0.0


def score_card(side, data):
    name = pretty_model_name(data['model'])
    per_round = clamp_pct(data['per_round_accuracy_pct'])
    majority = clamp_pct(data['majority_accuracy_pct'])
    return f'''
    <div class="ge-card">
      <div class="ge-kicker">{escape(side)}</div>
      <h3>{escape(name)}</h3>
      <div class="ge-muted">{escape(str(data['model']))}</div>
      <div class="ge-score">{per_round:.1f}%</div>
      <div class="ge-label">per-round accuracy</div>
      <div class="ge-bar"><span style="width:{per_round:.2f}%"></span></div>
      <div class="ge-mini-row"><span>correct samples</span><b>{int(data['correct'])}/{int(data['samples'])}</b></div>
      <div class="ge-mini-row"><span>majority vote</span><b>{majority:.1f}%</b></div>
      <div class="ge-bar ge-bar-alt"><span style="width:{majority:.2f}%"></span></div>
    </div>'''


def delta_card(label, delta):
    delta = float(delta)
    tone = 'pos' if delta > 0 else 'neg' if delta < 0 else 'flat'
    sign = '+' if delta > 0 else ''
    return f'''
    <div class="ge-card ge-delta ge-{tone}">
      <div class="ge-kicker">delta</div>
      <h3>{escape(label)}</h3>
      <div class="ge-score">{sign}{delta:.2f} pp</div>
      <div class="ge-muted">test model minus base model</div>
    </div>'''


def answer_distribution(answers, gold_letter):
    total = max(1, len(answers))
    chunks = []
    for answer, count in sorted(Counter(answers).items(), key=lambda kv: (-kv[1], str(kv[0]))):
        width = 100 * count / total
        cls = 'right' if answer == gold_letter else 'miss' if answer != '?' else 'unknown'
        chunks.append(
            f'<div class="ge-dist-row"><span class="ge-dist-answer ge-{cls}">{escape(str(answer))}</span>'
            f'<div class="ge-dist-track"><span class="ge-{cls}" style="width:{width:.2f}%"></span></div>'
            f'<b>{count}/{total}</b></div>'
        )
    return ''.join(chunks) or '<div class="ge-muted">No answers captured.</div>'


def round_pills(answers, hits, gold_letter):
    pills = []
    for idx, (answer, hit) in enumerate(zip(answers, hits), start=1):
        cls = 'right' if hit else 'miss' if answer != '?' else 'unknown'
        title = f'round {idx}: answer {answer}, gold {gold_letter}'
        pills.append(f'<span class="ge-pill ge-{cls}" title="{escape(title)}">R{idx}: {escape(str(answer))}</span>')
    return ''.join(pills) or '<span class="ge-muted">No rounds captured.</span>'


def model_slice(side, q):
    data = q[side]
    majority = data['majority_answer']
    majority_cls = 'right' if majority == q['gold_letter'] else 'miss' if majority != '?' else 'unknown'
    return f'''
    <div class="ge-model-slice">
      <div class="ge-model-head">
        <b>{escape(side)}</b>
        <span>majority: <b class="ge-{majority_cls}">{escape(str(majority))}</b></span>
        <span>hits: <b>{sum(data['hits'])}/{data['total']}</b></span>
      </div>
      <div class="ge-pills">{round_pills(data['answers'], data['hits'], q['gold_letter'])}</div>
      {answer_distribution(data['answers'], q['gold_letter'])}
    </div>'''


def render_question_slice(q):
    question = escape(q['question'][:VISUAL_TEXT_PREVIEW_CHARS])
    gold = escape(f"{q['gold_letter']}: {q['gold_text']}")
    return f'''
    <section class="ge-question">
      <div class="ge-question-head">
        <div class="ge-kicker">question slice {int(q['question_index'])}</div>
        <div class="ge-gold">gold answer: <b>{gold}</b></div>
      </div>
      <p>{question}</p>
      <div class="ge-model-grid">
        {model_slice('base', q)}
        {model_slice('test', q)}
      </div>
    </section>'''


def build_score_figure():
    score_df = pd.DataFrame([
        {'model': 'base', 'metric': 'per-round accuracy', 'pct': totals['base']['per_round_accuracy_pct']},
        {'model': 'test', 'metric': 'per-round accuracy', 'pct': totals['test']['per_round_accuracy_pct']},
        {'model': 'base', 'metric': 'majority accuracy', 'pct': totals['base']['majority_accuracy_pct']},
        {'model': 'test', 'metric': 'majority accuracy', 'pct': totals['test']['majority_accuracy_pct']},
    ])
    score_df['label'] = score_df['pct'].map(lambda v: f'{v:.1f}%')
    fig = px.bar(
        score_df,
        x='metric',
        y='pct',
        color='model',
        barmode='group',
        text='label',
        color_discrete_map={'base': PLOT_COLORS['base'], 'test': PLOT_COLORS['test']},
        title='Score snapshot: base vs test',
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_layout(
        yaxis_title='accuracy percent',
        xaxis_title='',
        yaxis_range=[0, 105],
        legend_title_text='model side',
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=70, r=30, b=50, l=50),
    )
    fig.update_yaxes(gridcolor='#e8eef5')
    return fig


def build_hit_heatmap():
    if detail_df.empty:
        return None
    heat_df = detail_df.copy()
    heat_df['row_label'] = heat_df['model_side'] + ' R' + heat_df['round'].astype(int).astype(str)
    row_order = []
    for side in ('base', 'test'):
        for round_idx in sorted(heat_df.loc[heat_df['model_side'] == side, 'round'].dropna().astype(int).unique()):
            row_order.append(f'{side} R{round_idx}')
    question_order = sorted(heat_df['question_index'].dropna().astype(int).unique())
    pivot = heat_df.pivot_table(index='row_label', columns='question_index', values='hit', aggfunc='max').reindex(row_order).reindex(question_order, axis=1)
    answer_pivot = heat_df.pivot_table(index='row_label', columns='question_index', values='extracted_answer', aggfunc='first').reindex(row_order).reindex(question_order, axis=1)
    hover = []
    for row in pivot.index:
        hover_row = []
        for col in pivot.columns:
            hit = pivot.loc[row, col]
            ans = answer_pivot.loc[row, col]
            status = 'correct' if hit == 1 else 'miss'
            hover_row.append(f'{row}<br>Q{int(col)}<br>answer: {ans}<br>{status}')
        hover.append(hover_row)
    fig = go.Figure(data=go.Heatmap(
        z=pivot.fillna(0).values,
        x=[f'Q{int(q)}' for q in pivot.columns],
        y=list(pivot.index),
        text=hover,
        hovertemplate='%{text}<extra></extra>',
        zmin=0,
        zmax=1,
        colorscale=[[0, PLOT_COLORS['miss']], [0.49, PLOT_COLORS['miss']], [0.5, PLOT_COLORS['right']], [1, PLOT_COLORS['right']]],
        colorbar=dict(tickvals=[0, 1], ticktext=['miss', 'hit']),
    ))
    fig.update_layout(
        title='Round-by-round hit map',
        xaxis_title='question slice',
        yaxis_title='model and round',
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=70, r=30, b=50, l=90),
    )
    return fig


def render_eval_dashboard():
    shown_questions = question_summaries[:int(VISUAL_MAX_QUESTIONS)]
    hidden = max(0, len(question_summaries) - len(shown_questions))
    gpu_plan = 'parallel: base on GPU 0, test on GPU 1' if parallel_two_gpu else f'sequential: device_map={DEVICE_MAP}'
    hidden_note = ''
    if hidden:
        hidden_note = f'<div class="ge-note">Showing the first {len(shown_questions)} question slices. {hidden} more are saved in the result files.</div>'
    html = f'''
    <style>
      .ge-wrap {{ font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: #18212f; }}
      .ge-wrap * {{ box-sizing: border-box; }}
      .ge-hero {{ border: 1px solid #d5dde8; border-radius: 8px; padding: 18px; background: linear-gradient(135deg, #f7fbff, #f7fff9); margin: 14px 0; }}
      .ge-hero h2 {{ margin: 0 0 8px; font-size: 24px; }}
      .ge-hero p {{ margin: 0; color: #44546a; }}
      .ge-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; margin: 12px 0; }}
      .ge-card {{ border: 1px solid #d5dde8; border-radius: 8px; padding: 14px; background: #ffffff; box-shadow: 0 1px 2px rgba(24, 33, 47, 0.06); }}
      .ge-card h3 {{ margin: 4px 0 6px; font-size: 16px; overflow-wrap: anywhere; }}
      .ge-kicker {{ color: #52647a; text-transform: uppercase; font-size: 11px; letter-spacing: .08em; font-weight: 700; }}
      .ge-muted {{ color: #66758a; font-size: 12px; overflow-wrap: anywhere; }}
      .ge-score {{ margin-top: 12px; font-size: 34px; line-height: 1; font-weight: 800; }}
      .ge-label {{ color: #52647a; font-size: 12px; margin: 2px 0 8px; }}
      .ge-bar, .ge-dist-track {{ height: 10px; border-radius: 6px; overflow: hidden; background: #e8eef5; }}
      .ge-bar span, .ge-dist-track span {{ display: block; height: 100%; background: #2f6fed; border-radius: 6px; }}
      .ge-bar-alt span {{ background: #19a974; }}
      .ge-mini-row {{ display: flex; justify-content: space-between; gap: 10px; margin-top: 8px; color: #334155; font-size: 13px; }}
      .ge-delta.ge-pos {{ border-color: #8fd8b0; background: #f2fff7; }}
      .ge-delta.ge-neg {{ border-color: #f1a1a1; background: #fff6f6; }}
      .ge-delta.ge-flat {{ border-color: #d5dde8; background: #fafcff; }}
      .ge-flow {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 8px; margin: 12px 0; }}
      .ge-step {{ border: 1px dashed #b9c6d6; border-radius: 8px; padding: 10px; background: #fbfdff; }}
      .ge-step b {{ display: block; margin-bottom: 4px; }}
      .ge-question {{ border: 1px solid #d5dde8; border-radius: 8px; padding: 14px; background: #ffffff; margin: 12px 0; }}
      .ge-question p {{ margin: 10px 0; line-height: 1.45; color: #243244; }}
      .ge-question-head {{ display: flex; justify-content: space-between; gap: 12px; flex-wrap: wrap; }}
      .ge-gold {{ color: #243244; }}
      .ge-model-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 12px; }}
      .ge-model-slice {{ border: 1px solid #e0e7ef; border-radius: 8px; padding: 12px; background: #fbfdff; }}
      .ge-model-head {{ display: flex; gap: 12px; justify-content: space-between; flex-wrap: wrap; font-size: 13px; margin-bottom: 8px; }}
      .ge-pills {{ display: flex; flex-wrap: wrap; gap: 6px; margin: 8px 0; }}
      .ge-pill {{ border-radius: 8px; padding: 4px 8px; font-size: 12px; font-weight: 700; border: 1px solid transparent; }}
      .ge-right {{ color: #047857; }}
      .ge-miss {{ color: #b42318; }}
      .ge-unknown {{ color: #8a6100; }}
      .ge-pill.ge-right {{ background: #e9fbf1; border-color: #8fd8b0; }}
      .ge-pill.ge-miss {{ background: #fff0f0; border-color: #f1a1a1; }}
      .ge-pill.ge-unknown {{ background: #fff8db; border-color: #edd37a; }}
      .ge-dist-row {{ display: grid; grid-template-columns: 34px 1fr 42px; align-items: center; gap: 8px; margin-top: 6px; font-size: 12px; }}
      .ge-dist-answer {{ font-weight: 800; }}
      .ge-dist-track .ge-right {{ background: #19a974; }}
      .ge-dist-track .ge-miss {{ background: #e5484d; }}
      .ge-dist-track .ge-unknown {{ background: #d5a100; }}
      .ge-note {{ color: #52647a; font-size: 13px; margin-top: 10px; }}
    </style>
    <div class="ge-wrap">
      <div class="ge-hero">
        <h2>Gemma 4 comparison dashboard</h2>
        <p>This view follows one evaluation slice from prompt window to model answers, extracted labels, majority vote, and final delta.</p>
      </div>
      <div class="ge-grid">
        <div class="ge-card"><div class="ge-kicker">task</div><h3>{escape(str(TASK))}</h3><div class="ge-muted">sample window starts at {int(samples_start)}</div></div>
        <div class="ge-card"><div class="ge-kicker">slice size</div><h3>{int(N_QUESTIONS)} questions x {int(ROUNDS)} rounds</h3><div class="ge-muted">{int(N_QUESTIONS) * int(ROUNDS)} samples per model</div></div>
        <div class="ge-card"><div class="ge-kicker">hardware plan</div><h3>{escape(gpu_plan)}</h3><div class="ge-muted">same questions, same sampling settings</div></div>
      </div>
      <div class="ge-grid">
        {score_card('base', totals['base'])}
        {score_card('test', totals['test'])}
        {delta_card('per-round accuracy', totals['delta_pp'])}
        {delta_card('majority accuracy', totals['majority_delta_pp'])}
      </div>
      <div class="ge-card">
        <div class="ge-kicker">how the data moves</div>
        <div class="ge-flow">
          <div class="ge-step"><b>1. Prompt slice</b><span>A small MMLU-Pro window is selected with samples_start and N_QUESTIONS.</span></div>
          <div class="ge-step"><b>2. Repeated rounds</b><span>Each model answers the same prompt ROUNDS times with sampled decoding.</span></div>
          <div class="ge-step"><b>3. Extracted answer</b><span>The evaluator reads the final A-J answer from each model response.</span></div>
          <div class="ge-step"><b>4. Vote and delta</b><span>Correct samples and majority votes become the comparison scores.</span></div>
        </div>
      </div>
      {''.join(render_question_slice(q) for q in shown_questions)}
      {hidden_note}
    </div>
    '''
    return html


if RUN_EVAL:
    dashboard_html = render_eval_dashboard()
    plotly_figures = [fig for fig in (build_score_figure(), build_hit_heatmap()) if fig is not None]
    plotly_html_sections = []
    display(HTML(dashboard_html))
    for idx, fig in enumerate(plotly_figures):
        fig.show()
        plotly_html_sections.append(fig.to_html(full_html=False, include_plotlyjs=(idx == 0)))
else:
    dashboard_html = ''
    plotly_figures = []
    plotly_html_sections = []
    print('No dashboard rendered because RUN_EVAL=False.')


## 7. Save report

In [ ]:
# Save results and print a readable report.

if RUN_EVAL:
    dashboard_html = globals().get('dashboard_html', '')
    plotly_html_sections = globals().get('plotly_html_sections', [])

    detail_path = run_dir / 'comparison_details.parquet'
    csv_path = run_dir / 'summary.csv'
    json_path = run_dir / 'summary.json'
    report_path = run_dir / 'report.md'

    html_path = run_dir / 'visual_report.html'

    detail_df.to_parquet(detail_path, index=False)
    pd.DataFrame([
        {'side': side, **values}
        for side, values in totals.items()
        if isinstance(values, dict)
    ]).to_csv(csv_path, index=False)
    json_path.write_text(json.dumps({
        'run_name': RUN_NAME,
        'task': TASK,
        'base_model': base_model,
        'test_model': test_model,
        'n_questions': N_QUESTIONS,
        'rounds': ROUNDS,
        'samples_start': samples_start,
        'totals': totals,
        'questions': question_summaries,
        'created_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    }, indent=2))

    lines = []
    lines.append(f'# Gemma 4 Kaggle Eval: {RUN_NAME}')
    lines.append('')
    lines.append(f'- task: `{TASK}`')
    lines.append(f'- base: `{base_model}`')
    lines.append(f'- test: `{test_model}`')
    lines.append(f'- window: samples_start `{samples_start}`, questions `{N_QUESTIONS}`, rounds `{ROUNDS}`')
    lines.append('')
    lines.append('| Side | Model | Samples | Correct | Per-round acc | Majority acc |')
    lines.append('|---|---|---:|---:|---:|---:|')
    for side in ('base', 'test'):
        t = totals[side]
        lines.append(f"| {side} | `{t['model']}` | {t['samples']} | {t['correct']} | {t['per_round_accuracy_pct']:.2f}% | {t['majority_accuracy_pct']:.2f}% |")
    lines.append('')
    lines.append(f"Per-round delta: **{totals['delta_pp']:+.2f} pp**")
    lines.append(f"Majority delta: **{totals['majority_delta_pp']:+.2f} pp**")
    lines.append('')
    for q in question_summaries:
        lines.append(f"## Q{q['question_index']}: {q['question'][:140]}")
        lines.append(f"Gold: **{q['gold_letter']}** {q['gold_text']}")
        lines.append(f"- base answers: `{q['base']['answers']}` hits {sum(q['base']['hits'])}/{q['base']['total']}")
        lines.append(f"- test answers: `{q['test']['answers']}` hits {sum(q['test']['hits'])}/{q['test']['total']}")
        lines.append('')
    report_path.write_text('\n'.join(lines))

    if dashboard_html:
        visual_doc = '<!doctype html><html><head><meta charset="utf-8"><title>Gemma 4 Eval Visual Report</title></head><body>' + dashboard_html + ''.join(plotly_html_sections) + '</body></html>'
        html_path.write_text(visual_doc)

    print('\nSaved files:')
    paths_to_show = [detail_path, csv_path, json_path, report_path]
    if dashboard_html:
        paths_to_show.append(html_path)
    for p in paths_to_show:
        print(f'  {p}')
    if dashboard_html:
        print('\nVisual dashboard was shown above and saved as visual_report.html.')
    else:
        print('\nRun the dashboard cell above to render the visual report.')
else:
    print('No results to save because RUN_EVAL=False.')
